# Lab 3: Statistical Significance and Permutation Testing

In this lab, you will investigate why a small p-value does not necessarily mean a large or useful effect. Then you will build a permutation test and use simulation to compare its power with a two-sample t-test.

By the end, you should be able to interpret effect size and uncertainty alongside statistical significance, explain why label shuffling is justified, and implement a power simulation that works for several data-generating distributions.

You may add cells to explore intermediate steps. Use the variable names specified in each question for your answers.

Run the setup cell before starting. It updates older package installations when needed. If setup asks you to restart, choose **Kernel → Restart Kernel**, then run the setup cell again before continuing.


In [1]:
source("lab3_packages.R")
if (lab3_ensure_packages()) {
  library(ottr)
  source("lab3_feedback.R")
}


Updating the lab's R packages. This may take a few minutes.




The downloaded binary packages are in
	/var/folders/7b/3n14tkpn43159_mb97d03mbm0000gn/T//Rtmpj0yWfB/downloaded_packages


Lab packages ready: ottr 1.6.0; testthat 3.3.2.

Warning message:
“package ‘ottr’ was built under R version 4.5.2”


## Part 1: Statistical Significance versus Effect Size

For a fixed nonzero effect in the normal model we study here, power approaches 1 as sample size grows. Even a very small effect can therefore produce a statistically significant result. What else do we need to decide whether that result matters?


### Screen Time Setting

A company studies changes in daily screen time after participants start using its app. A positive value means screen time decreased. Assume participants were randomly sampled and their observations are independent.

We test

$$H_0: \mu = 0 \quad\text{versus}\quad H_A: \mu \ne 0.$$

Suppose screen-time changes follow a normal distribution with population standard deviation $\sigma = 45$ minutes and true mean reduction $\mu = 3$ minutes per day.

This before-and-after setting lets us study a mean change. To establish that the app caused the change, we would also need a study design that addresses other possible causes.


Recall the discussion of statistical significance and effect size in Lecture #5, then answer the following.


**Question 1.1.** Which statements correctly describe p-values, sample sizes, and effect sizes in this normal model?

1. For a fixed nonzero effect and significance level, increasing sample size increases power.
2. Failing to reject $H_0$ proves that the effect is exactly zero.
3. A statistically significant p-value guarantees that the effect is practically important.
4. A confidence interval gives information about effect magnitude and uncertainty that a p-value alone does not provide.
5. For a fixed nonzero effect, power approaches 1 as sample size increases; any finite study can still fail to reject $H_0$.

Store all correct choices in `sig_eff_size`, in ascending order.


_Points:_ 0.5

In [ ]:
sig_eff_size <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_1.R")

**Reminder:** Power is the probability of rejecting the null when a specified alternative is true. A p-value describes how unusual the observed result is under the null. Neither quantity, by itself, establishes practical importance.


We will simulate $n = 2{,}000$ observations from $\mathcal{N}(3,45^2)$. Normal observations give an exact one-sample t-test under the null. With sufficiently large samples and suitable conditions, the test can also be approximately valid for non-normal observations.

Run the next cell to generate the data and compare its histogram with the population density.


In [ ]:
set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!
app_large_sample <- rnorm(n=2000, mean=3, sd=45)
par(bg='white')
hist(app_large_sample, main="Distribution of Simulated Screen Time Reduction Sample", 
    xlab="Screen Time Reduction", breaks=30, freq=FALSE, col='royalblue',
    xlim = range(app_large_sample))
curve(dnorm(x, mean=3, sd=45), add=TRUE, col='red', lwd=5)

It's not perfect, but it looks like our sample is a pretty good representation of the distribution. 

**Question 1.2.1.** Write `one_sample_test(my_data, mu_0=0, conf_level=0.95)` to perform a two-sided one-sample t-test. Return the test result as an object with named components `p.value` and `conf.int`; the result returned by R's test function has these components.

Your function must work with other datasets, null means, and confidence levels. Apply it to `app_large_sample` for the company's hypothesis and store the p-value in `p_val_large_n`.


_Points:_ 1

In [ ]:
one_sample_test <- function(my_data, mu_0=0, conf_level=0.95) {
  # YOUR CODE HERE
}
p_val_large_n <- NULL # YOUR CODE HERE
p_val_large_n


In [ ]:
. = ottr::check("tests/q1_2_1.R")

**Question 1.2.2.** Use your `one_sample_test()` function to obtain the 95% confidence interval for the population mean from `app_large_sample`. Store it in `ci_large_n`, a numeric vector of length 2. The function should also return the appropriate interval when the data or confidence level changes.


_Points:_ 1

In [ ]:
ci_large_n <- NULL # YOUR CODE HERE
ci_large_n


In [ ]:
. = ottr::check("tests/q1_2_2.R")

**Question 1.3.** Which interpretation agrees with your p-value and 95% confidence interval?

1. We reject $H_0$ at $\alpha=0.05$, and the interval supports an average reduction exceeding 10 minutes.
2. We reject $H_0$ at $\alpha=0.05$, but the interval supports an average reduction below 10 minutes; practical importance depends on what reduction would be worthwhile.
3. We fail to reject $H_0$ at $\alpha=0.05$, so we have proved that the mean reduction is zero.
4. We fail to reject $H_0$ at $\alpha=0.05$, but the interval supports an average reduction exceeding 10 minutes.

Assign your choice to `interp_large_n`.


_Points:_ 1

In [ ]:
interp_large_n <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_3.R")

**Question 1.4.1.** Repeat the procedure from Question 1.2.1 for a much smaller sample. Write `draw_app_sample(n, mean=3, sd=45)` to generate independent normal observations with the requested sample size, population mean, and population standard deviation. The function must work with other positive sample sizes and standard deviations, and with other means. Set the random seed before calling it, rather than inside the function.

Use your function to generate 30 observations from $\mathcal{N}(3,45^2)$, storing them in `app_small_sample`. Apply your `one_sample_test()` function to test the same hypotheses and store its p-value in `p_val_small_n`.

The coding points assess your sample generator on additional inputs. The small-sample test application is checked for feedback; your reusable test function was assessed in Question 1.2.


_Points:_ 1

In [ ]:
draw_app_sample <- function(n, mean=3, sd=45) {
  # YOUR CODE HERE
}
set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!
# YOUR CODE HERE
p_val_small_n


In [ ]:
. = ottr::check("tests/q1_4_1.R")

**Question 1.4.2.** The two simulated samples have different observed means as well as different sizes. To isolate the role of sample size, imagine holding the observed mean and standard deviation fixed. Why would the larger sample give stronger evidence against $H_0$?

1. Increasing sample size automatically increases the observed effect size.
2. Increasing sample size automatically increases the population standard deviation.
3. The standard error $s/\sqrt{n}$ decreases, so the same nonzero observed mean is farther from the null value in standard-error units.
4. Changing sample size has no effect on the test statistic or its p-value.

Assign your choice to `p_val_comparison`. The actual p-values also reflect the different observed means in these two samples.


_Points:_ 0.5

In [ ]:
p_val_comparison <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q1_4_2.R")

**Question 1.5.1.** The company wants to detect a true average reduction of 3 minutes per day when the population standard deviation is 45 minutes. Determine the smallest sample size giving at least 80% power for a two-sided one-sample t-test at significance level 0.05.

Write two reusable functions: `one_sample_power(n, delta, sd, alpha=0.05)`, returning theoretical t-test power, and `sample_size_for_power(delta, sd, target_power=0.8, alpha=0.05)`, returning the smallest integer sample size achieving the target. Both functions must work for other positive effect sizes, standard deviations, and significance levels; the second must also work for other power targets.

Apply your functions to this setting and store the integer sample size in `n_required`. Verify that it achieves the target while one fewer observation does not, storing those power values in `power_at_required` and `power_one_fewer`.

*Hint:* Revisit the poker example in Lecture #5. Be careful about whether that example used the same alternative hypothesis as this one.


_Points:_ 1.5

In [ ]:
one_sample_power <- function(n, delta, sd, alpha=0.05) {
  # YOUR CODE HERE
}
sample_size_for_power <- function(delta, sd, target_power=0.8, alpha=0.05) {
  # YOUR CODE HERE
}
# YOUR CODE HERE
c(n_required=n_required, power_at_required=power_at_required, power_one_fewer=power_one_fewer)


In [ ]:
. = ottr::check("tests/q1_5_1.R")

**Question 1.5.2.** Suppose the true reduction is only $\delta=0.5$ minutes (30 seconds). Use your `one_sample_power()` function to compute the power of a two-sided one-sample t-test with $n=5{,}000$, $\sigma=45$, and $\alpha=0.05$. Store the result in `power_tiny_effect`.

Then compute the sample size needed for 80% power, or use your power result to assess these choices:

1. About 500 participants would be sufficient.
2. About 5,000 participants would be sufficient.
3. Roughly 64,000 participants would be needed; the company should weigh the cost against the value of detecting a 30-second reduction.
4. Power is independent of sample size.

Store your choice in `power_tiny_conclusion`.


_Points:_ 1.5

In [ ]:
power_tiny_effect <- NULL # YOUR CODE HERE
power_tiny_effect
power_tiny_conclusion <- NULL # YOUR CODE HERE
power_tiny_conclusion


In [ ]:
. = ottr::check("tests/q1_5_2.R")

As we saw both in Lab \#2 and in Lectures \#4 and \#5, we can plot power curves to understand how the power changes for each true effect size $\delta$ as the sample size increases. Given some of the answers to the previous questions, it should come as no surprise that the power curve when $\delta = 0.5$ remains well below the power curve when $\delta = 3$. Run the cell to see for yourself by just how much the two power curves differ, especially at small sample sizes. (The grey dotted line in the first plot marks our desired 80% power.)

**Power convention:** The supplied curves and reference calculations use the default `power.t.test()` convention used in lecture. For a two-sided test, the default `strict=FALSE` omits rejection in the tail opposite the true effect; `strict=TRUE` includes both rejection tails. The grader accepts either convention in Questions 1.5.1–1.5.2. Use your chosen convention consistently when checking the required sample size.


In [ ]:
# Define parameters for plots
n_values <- seq(from=100, to=20000, length.out=100)
deltas <- c(0.5, 1, 1.5, 2, 3)
sd_val <- 45

# Plot 1 (standard power curves)
par(bg='white')
plot(NULL, xlim = range(n_values), ylim = c(0, 1),
     xlab = "Sample Size (n)", ylab = "Statistical Power",
     main = expression(paste("Power Curves with ", sigma == 45, ", ", H[0], ": ", mu == 0)))
colors <- rainbow(length(deltas))
for (i in seq_along(deltas)) {
  # Calculate power for each n in the range for the current delta
  powers <- sapply(n_values, function(n) {
    power.t.test(n = n, delta = deltas[i], sd = sd_val, 
                 sig.level = 0.05, type = "one.sample")$power
  })
  # Add the line to the plot
  lines(n_values, powers, col = colors[i], lwd = 2)
}
# Add legend to plot
legend_labels <- sapply(deltas, function(d) as.expression(bquote(delta == .(d))))
legend("bottomright",  legend = legend_labels,  col = colors, lwd = 2, 
       title = "True Effect Size")
abline(h = 0.8, lty = 2, col = "darkgrey")   # Add horizontal line for 80% power

# Plot 2 (power gaps)
par(bg='white')
plot(NULL, xlim = range(n_values), ylim = c(0, 1),
     xlab = "Sample Size (n)", ylab = "Power Difference",
     main = expression(paste("Gap in Power from ", delta == 3)))
# Pre-calculate baseline power for delta = 3
baseline_p <- sapply(n_values, function(n) {
  power.t.test(n = n, delta = 3, sd = sd_val, type = "one.sample")$power
})
for (i in 1:(length(deltas)-1)) {
  current_p <- sapply(n_values, function(n) {
    power.t.test(n = n, delta = deltas[i], sd = sd_val, type = "one.sample")$power
  })
  lines(n_values, baseline_p - current_p, col = colors[i], lwd = 2)
}
# Add legend to plot
legend("topright", legend = legend_labels[1:4], col = colors[1:4], 
        lwd = 2, cex = 0.7, title = "True Effect Size")


**Question 1.6.** Lecture #5 emphasizes that practical importance depends on the setting. As an extension, we will also compute a standardized effect size: the magnitude of the one-sample Cohen's d,

$$d=\frac{|\overline{x}-\mu_0|}{s}.$$

Reference values of 0.2, 0.5, and 0.8 are commonly described as small, medium, and large. For this exercise, classify values below 0.2 as `"very small"`, values from 0.2 up to 0.5 as `"small"`, values from 0.5 up to 0.8 as `"medium"`, and values at least 0.8 as `"large"`. These are rough reference points, not measures of practical importance.

For example, the poker calculation in Lecture #5 uses an effect of approximately \$0.1023 per hand and an SD of \$7.1812, giving $d\approx0.0142$. The lecture still treats ten cents per hand as potentially worthwhile because the gain accumulates over many hands.

Implement `cohens_d_func(my_data, mu_0)`. Return `NA` when the sample standard deviation is zero or undefined; otherwise return `c(d, classification)`. Combining a number and text produces a character vector, so use `as.numeric()` when reading its first entry.

Apply your function to `app_large_sample` with `mu_0=0` and store the result in `cohens_d_result`.


_Points:_ 2

In [ ]:
cohens_d_func <- function(my_data, mu_0) {
  # YOUR CODE HERE
}
cohens_d_result <- NULL # YOUR CODE HERE
cohens_d_result


In [ ]:
. = ottr::check("tests/q1_6.R")

Part 1 is complete. We now turn to the two-sample testing and permutation ideas from Lecture #6.


## Part 2: Permutation Testing and Power

In Lecture #6, you built a permutation test by shuffling labels or outcomes and recomputing the absolute difference in group means. We will extend that procedure to another statistic and compare power under several population distributions.

A permutation test does not require normally distributed observations, but shuffling needs a justification. **Exchangeability** means that, under the null model, reassigning the group labels does not change the joint distribution of the observations: the labels contain no information about which population an observation came from. In our simulated setting, the independent groups have the same population distribution when the mean shift is zero, so their labels are interchangeable. Equal means alone would not justify this if the groups had different distribution shapes or variances.


**Question 2.1.** Welch's two-sample $t$-test is the standard approach for comparing the means of two independent groups since it does not assume the groups have equal variances. Which of the following conditions support the approximate validity of this test? 

1. The observations in each group are normally distributed, or the sample size is large enough for an appropriate normal approximation to the sample mean.
2. The two groups must have the same sample size (i.e., $n_1 = n_2$).
3. The true population standard deviation $\sigma$ must be the same in both groups.
4. The observations must be independent within each group and independent between the two groups.
5. The data must be collected through a random assignment of participants to each of the groups. 

Select all that are correct and store your choice(s) as a vector `ttest_conditions`. If multiple options are correct, put them in the vector in ascending order (smallest to largest).

_Points:_ 0.5

In [ ]:
ttest_conditions <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_1.R")

**Reminder:** Welch's test allows unequal population variances. Independence matters, and normal approximations can be unreliable for very small samples from highly skewed or heavy-tailed populations.


For the first new context, in which we explore another test statistic, we will use the same fictional dataset from Lecture \#6. Run the cell below to create it.

In [ ]:
# Run this cell -- do not modify!
set.seed(4162026)
hand_code_df <- data.frame(
  score = c(2, 5, 3, -3, 8, 9, 10, -1, 14, 6),
  group = c(rep("handwritten", 5), rep("coding", 5))
)

**Question 2.2.** If $\mu_\mathrm{coding}$ denotes the true mean of those who were tasked with doing the coding exercises and $\mu_\mathrm{handwritten}$ denotes the true mean of those who were tasked with doing the hand-written exercises, run a two-sample $t$-test on `hand_code_df` to test: 

$$
H_0: \mu_\mathrm{coding} = \mu_\mathrm{handwritten} \quad\text{vs.}\quad H_A: \mu_\mathrm{coding} \neq \mu_\mathrm{handwritten}
$$

Write `two_sample_p(df)` to return the two-sided Welch t-test p-value. As in the next question, assume the first column contains numeric observations and the second contains two group labels; your function must work with other labels and unequal group sizes. Apply it to `hand_code_df` and store the p-value in `p_val_ttest`.

_Points:_ 0.5

In [ ]:
two_sample_p <- function(df) {
  # YOUR CODE HERE
}
p_val_ttest <- NULL # YOUR CODE HERE
p_val_ttest


In [ ]:
. = ottr::check("tests/q2_2.R")

**Question 2.3.** In Your Turn #2 of Lecture #6, you used the absolute difference in group means for a permutation test. Write `diff_in_means(df)`, which returns that statistic, then apply it to `hand_code_df` to obtain `obs_test_stat`.

Assume the first column of `df` contains numeric observations and the second contains group labels. You can access column `i` with `df[[i]]`. Your function should work with different two-group labels and observations.

*Hint:* `tapply(values, groups, f)` applies `f` separately to each group, and `unname()` removes an object's name attribute.


_Points:_ 1

In [ ]:
diff_in_means <- function(df) {
  # YOUR CODE HERE
}

# Apply your function to the original data to get the observed test statistic
obs_test_stat <- NULL # YOUR CODE HERE
obs_test_stat

In [ ]:
. = ottr::check("tests/q2_3.R")

**Question 2.4.** Adapt the permutation procedure from Your Turn #2 of Lecture #6 into `permute_stats(df, stat_fn, n_perms=5000)`. Return a numeric vector containing the statistic from each of `n_perms` independent random shuffles, using `stat_fn` to determine the statistic values. Assume the first dataframe column contains observations and the second contains two group labels.

The function must work with other data, group sizes, statistic functions, and shuffle counts. You may shuffle labels or scores; otherwise preserve the observations and group sizes. Set seeds before calls, rather than inside the function.

Use it with `hand_code_df` and `diff_in_means()` to perform 5,000 shuffles, storing the results in `perm_stats`. As in lecture, estimate the p-value as the proportion of shuffled statistics greater than or equal to `obs_test_stat`, counting ties as extreme. Store it in `p_val_perm`.


In [ ]:
permute_stats <- function(df, stat_fn, n_perms=5000) {
  # YOUR CODE HERE
}
set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!
perm_stats <- NULL # YOUR CODE HERE
p_val_perm <- NULL # YOUR CODE HERE
p_val_perm


Run the following code cell to visualize the null distribution from your permutation test. The red vertical line marks the observed test statistic, `obs_test_stat`. Take a second to think about what you observe.

_Points:_ 1.5

In [ ]:
# Run this cell -- do not modify!
par(bg='white')
hist(perm_stats, col="lightblue", main="Permutation Null Distribution", xlab="Absolute Difference in Group Means")
abline(v=obs_test_stat, col="red", lwd=2)
legend("topright", legend="Observed statistic", col="red", lwd=2)

In [ ]:
. = ottr::check("tests/q2_4.R")

**Question 2.5.** Compare `p_val_perm` with `p_val_ttest`. Which statement best describes what you observed? 

1. The permutation test gives a substantially smaller $p$-value, showing it is much more powerful than the $t$-test.
2. The permutation test gives a substantially larger $p$-value, showing it is much less powerful than the $t$-test.
3. The two $p$-values are close in magnitude and both tests lead to the same decision about the null hypothesis $H_0$ at $\alpha = 0.05$.
4. The two $p$-values are identical because the permutation test and the two-sample $t$-test are mathematically equivalent. 

Assign your answer (1, 2, 3, or 4) to `p_val_check` below. 

_Points:_ 0.5

In [ ]:
p_val_check <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_5.R")

**Question 2.6.** One of the appealing features of the permutation test is that it is not tied to any particular test statistic. Indeed, we could have chosen to use the median instead of the mean. Fill in the cell below by doing the following:

1. Fill in the function `diff_in_medians` which takes as input a dataframe `df` and returns the absolute difference in group medians. (Use `diff_in_means` as a reference.)
2. Apply this function on `hand_code_df` and obtain an observed test statistic `obs_median_stat` for our permutation test.
3. Reuse `permute_stats()` from **Question 2.4** with `diff_in_medians` to compute 5,000 shuffled test statistics.
4. Store the new permuted test statistics in `perm_median_stats`, and put the permutation $p$-value, computed as the proportion of shuffled statistics at least as extreme as the observed statistic, in `p_val_medians`, counting ties as extreme.

The coding points assess your median statistic on additional inputs. Reusing the permutation function is checked for feedback; its general shuffling behavior was assessed in Question 2.4.


In [ ]:
diff_in_medians <- function(df) {
  # YOUR CODE HERE
}
obs_median_stat <- NULL # YOUR CODE HERE
obs_median_stat

set.seed(123) # DO NOT DELETE OR CHANGE THIS LINE!
perm_median_stats <- NULL # YOUR CODE HERE
p_val_medians <- NULL # YOUR CODE HERE
p_val_medians


Run the following cell to see the null distribution of the permuted absolute difference in group medians. You should expect to see a histogram whose bins are much more disjoint due to the more disjoint nature of the test statistic. 

_Points:_ 1.5

In [ ]:
# Run this cell -- do not modify!
par(bg='white')
hist(perm_median_stats, col="lightgreen", main="Permutation Null Distribution", xlab="Absolute Difference in Group Medians")
abline(v=obs_median_stat, col="red", lwd=2)
legend("topright", legend="Observed statistic", col="red", lwd=2)

In [ ]:
. = ottr::check("tests/q2_6.R")

You have now implemented the permutation procedure using both means and medians. The shuffling logic stays the same when the statistic changes.

The optional `perm_test()` helper below can wrap `permute_stats()` and return the proportion of shuffled statistics at least as extreme as the observed statistic. Alternatively, use `permute_stats()` or your earlier shuffling code directly in Question 2.7. Set seeds before simulation calls rather than inside a helper.


In [ ]:
# Optional helper; completing it is not required if you implement the test in Q2.7.
perm_test <- function(df, stat_fn, n_perms=500) {
  # Your implementation, if you choose to use this helper.
}


In Lecture #4, Your Turn #2, you packaged a power simulation into a reusable function. We will use the same idea here: simulate a new study, run a test, and count how often the null is rejected.

The supplied `sim_power_t_two()` below does this for a two-sample t-test. Here `n` is the sample size per group, `delta` is the difference in population means, `sd` is the population standard deviation, and `reps` is the number of simulated studies.

Its `draw_group` argument determines how observations are generated. The default `draw_normal(n, mean, sd)` generates normal observations. Later, you will supply your own generators while reusing the testing functions. In these equal-variance simulations we use the pooled t-test (`var.equal=TRUE`); the earlier observed-data example used Welch's test.


In [ ]:
# Supplied functions: run this cell.
draw_normal <- function(n, mean, sd) rnorm(n, mean=mean, sd=sd)

sim_power_t_two <- function(n, delta, sd, alpha=0.05, reps=500,
                            draw_group=draw_normal) {
  rejections <- replicate(reps, {
    group1 <- draw_group(n, mean=0, sd=sd)
    group2 <- draw_group(n, mean=delta, sd=sd)
    t.test(group1, group2, var.equal=TRUE)$p.value < alpha
  })
  mean(rejections)
}

set.seed(123)
t_power_estimate <- sim_power_t_two(n=15, delta=4.5, sd=4, reps=100)
t_power_estimate


**Question 2.7.** Write `sim_power_perm(n, delta, sd, alpha=0.05, reps=500, n_perms=500, draw_group=draw_normal)` to estimate the power of a permutation test using the absolute difference in group means.

For each of `reps` studies, generate independent groups of size `n` with population means 0 and `delta`, both with population standard deviation `sd`. Use `draw_group(n, mean, sd)` to generate each group. Perform a permutation test with `n_perms` shuffles and record whether its p-value is below `alpha`. Return the proportion of studies that reject the null.

Assume `n`, `reps`, and `n_perms` are positive integers, including 1; `sd > 0`; and `0 <= alpha <= 1`. The shift `delta` may be positive, zero, or negative.

Your function must work with different values of its arguments and with other generators that use the same interface. You may complete and use the optional `perm_test()` helper, or implement the permutation procedure directly. The supplied t-test function and the Lecture #4 power simulation are references for the outer repetition.

First test your code with the small debugging call below. That call checks execution rather than producing a reliable power estimate. Runtime depends on the computer, so a slow run alone does not indicate an error.


_Points:_ 2

In [ ]:
# Write the full function here.
# YOUR CODE HERE

# Debugging run: use it to check execution.
set.seed(123)
debug_perm_power <- sim_power_perm(n=15, delta=4.5, sd=4, reps=10, n_perms=20)
debug_perm_power

# Modest demonstration; larger reference curves are supplied later.
set.seed(123)
perm_power_estimate <- sim_power_perm(n=15, delta=4.5, sd=4, reps=100, n_perms=99)
perm_power_estimate


In [ ]:
. = ottr::check("tests/q2_7.R")

Next, keep the testing procedures fixed and change the data-generating distribution. Your general functions should allow this without copying the simulation loops.


In [ ]:
# A quick check of both functions; these are not the final power curves.
set.seed(321)
quick_normal <- c(
  t_test=sim_power_t_two(n=15, delta=4.5, sd=4, reps=10),
  permutation=sim_power_perm(n=15, delta=4.5, sd=4, reps=10, n_perms=20)
)
quick_normal


The quick estimates above may vary considerably. Use them to check execution, then use the larger reference simulation for interpretation.


In [ ]:
# Reference power curves will be plotted together after Question 2.8.


**Question 2.8.1.** Write `draw_skew(n, mean, sd)` to generate `n` observations from a shifted exponential distribution with the requested population mean and standard deviation. Your function should work for different means and positive values of `sd`.

Use this generator with both power functions to estimate power for `n=15`, `delta=4.5`, `sd=4`, and `alpha=0.05`. Use 100 simulated studies and 99 shuffles for the permutation test. Store your estimates in `t_power_skew` and `perm_power_skew`.

*Hint:* The mean and standard deviation of an exponential distribution are both `1/rate`. A location shift changes its mean but not its standard deviation.

The coding points assess your generator on additional parameters. The displayed power estimates help you compare the tests; a single numerical estimate cannot establish that a simulation was implemented correctly.


_Points:_ 1

In [ ]:
set.seed(123)
# Write your generator and the calls to estimate power.
# YOUR CODE HERE
c(t_test=t_power_skew, permutation=perm_power_skew)


In [ ]:
. = ottr::check("tests/q2_8_1.R")

**Question 2.8.2.** Write `draw_heavy(n, mean, sd, df_t=3)` to generate `n` observations from a shifted and scaled t distribution with the requested population mean and standard deviation. Your function should work for different means, positive values of `sd`, and `df_t > 2`.

Use it in both power functions with the same settings as Question 2.8.1 and `df_t=3`. Store your estimates in `t_power_heavy` and `perm_power_heavy`.

*Hint:* A standard t distribution with `df_t` degrees of freedom has standard deviation `sqrt(df_t/(df_t-2))`. Determine the scaling and shift needed to match the requested parameters.

The coding points assess your generator on additional parameters. The displayed power estimates help you compare the tests; a single numerical estimate cannot establish that a simulation was implemented correctly.


_Points:_ 1

In [ ]:
set.seed(123)
# Write your generator and the calls to estimate power.
# YOUR CODE HERE
c(t_test=t_power_heavy, permutation=perm_power_heavy)


In [ ]:
. = ottr::check("tests/q2_8_2.R")

### Compare the tests within each distribution

The next cell loads instructor-generated reference curves from `power_curves_reference.csv`. Each point used 300 independent studies, a mean shift of 4.5, population standard deviation 4, and 500 shuffles per permutation test, using the p-value calculation from Lecture #6. Both tests used the same dataset within each study.

These curves are simulation estimates. Small gaps may reflect variation from the finite number of studies, so compare overall patterns rather than treating each small difference as evidence that one test is better.


In [ ]:
# Supplied reference curves: run this cell.
n_vals <- c(5, 10, 15, 20, 25, 30)
power_curves <- read.csv("power_curves_reference.csv")



In [ ]:
# Run this cell to compare both tests in each distribution.
par(bg="white", mfrow=c(1,3), mar=c(4,4,3,1))
for (label in c("Normal","Skewed","Heavy_tailed")) {
  plot(NA, xlim=range(n_vals), ylim=c(0,1), xlab="n per group", ylab="Estimated power",
       main=gsub("_"," ",label))
  for (test_name in c("t_test","permutation")) {
    z <- subset(power_curves, distribution == label & test == test_name)
    color <- if (test_name == "t_test") "royalblue" else "firebrick"
    point <- if (test_name == "t_test") 15 else 19
    lines(z$n,z$power,type="b",pch=point,col=color,lwd=2)
  }
  legend("bottomright",c("t-test","Permutation"),col=c("royalblue","firebrick"),
         pch=c(15,19),lwd=2,bty="n",cex=0.8)
}
par(mfrow=c(1,1))


**Question 2.9.** Compare the two tests at the same sample size within each distribution. Which conclusion is supported by this simulation?

1. A permutation test has the same power for every data distribution because it does not require normality.
2. A small gap between two simulated estimates proves that the test with the higher estimate is always better.
3. Increasing the number of label shuffles always has the same effect on power as collecting more participants.
4. The tests have broadly similar power in these simulated settings. Small gaps may reflect simulation variation, and the results do not establish equivalence in every setting.

Assign your choice to `combined_power_conclusion`.


_Points:_ 0.5

In [ ]:
combined_power_conclusion <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_9.R")

## Congratulations, you are finished!!

To submit your assignment:

1. Select `Kernel -> Restart Kernel and Run All Cells...` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything is fine and all tests passed.
3. Download your notebook using `File -> Download`, then upload your notebook to Gradescope.
4. Stick around while the Gradescope autograder grades your work. Make sure you see that all tests have passed on Gradescope.
5. Check that you have a confirmation email from Gradescope and save it as proof of your submission.